# Recipe Recommendation Engine — Testing & Tuning

Interactive notebook for testing the heuristic scoring engine, tuning weights,
and visualizing score distributions.

In [ ]:
# Setup
import os
import json
import math
from datetime import datetime, timedelta
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from supabase import create_client, Client
from IPython.display import display, HTML

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to Supabase
SUPABASE_URL = os.environ.get('SUPABASE_URL', os.environ.get('NEXT_PUBLIC_SUPABASE_URL', ''))
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY', '')

assert SUPABASE_URL, 'Set SUPABASE_URL or NEXT_PUBLIC_SUPABASE_URL env var'
assert SUPABASE_KEY, 'Set SUPABASE_SERVICE_ROLE_KEY env var'

sb: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f'Connected to {SUPABASE_URL}')

## 4.2 Data Exploration

In [ ]:
# Set test user ID — change this to a real user in your DB
TEST_USER_ID = os.environ.get('TEST_USER_ID', 'YOUR_USER_ID_HERE')

# Count recipes
recipes_resp = sb.table('recipes').select('id', count='exact').is_('deleted_at', 'null').execute()
total_recipes = recipes_resp.count

# Count ingredients
ingredients_resp = sb.table('standardized_ingredients').select('id', count='exact').execute()
total_ingredients = ingredients_resp.count

# Count pantry items for test user
pantry_resp = sb.table('pantry_items').select('id', count='exact').eq('user_id', TEST_USER_ID).execute()
total_pantry = pantry_resp.count

print(f'Total recipes: {total_recipes}')
print(f'Total standardized ingredients: {total_ingredients}')
print(f'Pantry items for test user: {total_pantry}')

In [ ]:
# Distribution of ingredient counts per recipe
ri_resp = sb.table('recipe_ingredients').select('recipe_id').is_('deleted_at', 'null').execute()
ri_df = pd.DataFrame(ri_resp.data)
if not ri_df.empty:
    counts = ri_df.groupby('recipe_id').size().reset_index(name='ingredient_count')
    fig, ax = plt.subplots()
    ax.hist(counts['ingredient_count'], bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Number of Ingredients')
    ax.set_ylabel('Number of Recipes')
    ax.set_title('Distribution of Ingredient Counts per Recipe')
    plt.tight_layout()
    plt.show()
    print(counts['ingredient_count'].describe())
else:
    print('No recipe_ingredients data found')

In [ ]:
# Pantry coverage: what % of each recipe's ingredients does the test user have?
pantry_items = sb.table('pantry_items').select('standardized_ingredient_id').eq('user_id', TEST_USER_ID).execute()
pantry_ids = set(p['standardized_ingredient_id'] for p in pantry_items.data if p.get('standardized_ingredient_id'))

ri_all = sb.table('recipe_ingredients').select('recipe_id, standardized_ingredient_id').is_('deleted_at', 'null').execute()
ri_all_df = pd.DataFrame(ri_all.data)

if not ri_all_df.empty and pantry_ids:
    ri_all_df['in_pantry'] = ri_all_df['standardized_ingredient_id'].isin(pantry_ids)
    coverage = ri_all_df.groupby('recipe_id').agg(
        total=('standardized_ingredient_id', 'count'),
        matched=('in_pantry', 'sum')
    )
    coverage['ratio'] = coverage['matched'] / coverage['total']

    fig, ax = plt.subplots()
    ax.hist(coverage['ratio'], bins=20, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Pantry Coverage Ratio')
    ax.set_ylabel('Number of Recipes')
    ax.set_title('How much of each recipe can the test user make?')
    ax.axvline(0.4, color='red', linestyle='--', label='Min match threshold (0.4)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'Recipes with >=40% coverage: {(coverage["ratio"] >= 0.4).sum()} / {len(coverage)}')
else:
    print('Not enough data to compute coverage')

## 4.3 Scoring Simulation (Python port)

In [ ]:
# Constants (mirroring lib/recipe-rec/constants.ts)
PANTRY_STAPLES = {
    'salt', 'black pepper', 'pepper', 'water', 'cooking oil',
    'vegetable oil', 'olive oil', 'sugar', 'all-purpose flour', 'flour',
    'butter', 'garlic', 'onion', 'baking soda', 'baking powder',
    'vanilla extract', 'soy sauce', 'vinegar', 'rice', 'eggs', 'milk',
}

SUBSTITUTION_MAP = {
    'lemon': [('lime', 0.9)],
    'lime': [('lemon', 0.9)],
    'butter': [('margarine', 0.85), ('coconut oil', 0.7)],
    'margarine': [('butter', 0.85)],
    'heavy cream': [('coconut cream', 0.7), ('half and half', 0.75)],
    'milk': [('oat milk', 0.8), ('almond milk', 0.75)],
    'sugar': [('honey', 0.8), ('maple syrup', 0.75)],
    'vegetable oil': [('canola oil', 0.95), ('olive oil', 0.8)],
    'soy sauce': [('tamari', 0.9), ('coconut aminos', 0.7)],
}

DEFAULT_WEIGHTS = {
    'ingredientMatch': 0.40,
    'quantitySufficiency': 0.10,
    'expiryUrgency': 0.10,
    'pantryStaple': 0.05,
    'substitution': 0.10,
    'preference': 0.10,
    'popularity': 0.05,
    'diversity': 0.10,
}

STAPLE_PENALTY_FACTOR = 0.2
SUBSTITUTION_CREDIT_FACTOR = 0.8

print('Constants loaded')

In [ ]:
def score_recipe(recipe_id, recipe_title, ingredient_names, pantry_names, pantry_expiring,
                 cuisine=None, rating_avg=None, rating_count=None, tags=None,
                 user_cuisine_prefs=None, user_dietary_prefs=None,
                 recent_recipe_ids=None, recent_cuisines=None,
                 weights=None):
    """Python port of the TS scorer for prototyping."""
    w = weights or DEFAULT_WEIGHTS
    tags = tags or []
    user_cuisine_prefs = user_cuisine_prefs or []
    user_dietary_prefs = user_dietary_prefs or []
    recent_recipe_ids = recent_recipe_ids or set()
    recent_cuisines = recent_cuisines or []
    pantry_set = set(n.lower() for n in pantry_names)
    pantry_expiring_set = set(n.lower() for n in pantry_expiring)
    
    total = len(ingredient_names)
    if total == 0:
        return {'recipe_id': recipe_id, 'title': recipe_title, 'total_score': 0, 'match_ratio': 0}
    
    matched = [n for n in ingredient_names if n.lower() in pantry_set]
    missing = [n for n in ingredient_names if n.lower() not in pantry_set]
    
    # 1. Ingredient match
    match_ratio = len(matched) / total
    
    # 2. Quantity sufficiency (simplified — assume 0.5 for all)
    qty_score = 0.5 if matched else 0
    
    # 3. Expiry urgency
    expiring_used = sum(1 for n in ingredient_names if n.lower() in pantry_expiring_set)
    expiry_score = expiring_used / max(len(pantry_expiring_set), 1) if pantry_expiring_set else 0
    
    # 4. Staple adjustment
    staple_adj = sum((1 - STAPLE_PENALTY_FACTOR) / total for n in missing if n.lower() in PANTRY_STAPLES)
    
    # 5. Substitution credit
    sub_credit = 0
    non_staple_missing = [n for n in missing if n.lower() not in PANTRY_STAPLES]
    for name in non_staple_missing:
        subs = SUBSTITUTION_MAP.get(name.lower(), [])
        matching_subs = [conf for sub, conf in subs if sub.lower() in pantry_set]
        if matching_subs:
            sub_credit += max(matching_subs) * SUBSTITUTION_CREDIT_FACTOR / total
    
    # 6. Preference alignment
    pref_score = 0.5
    if cuisine and user_cuisine_prefs:
        if cuisine.lower() in [c.lower() for c in user_cuisine_prefs]:
            pref_score += 0.3
    tag_set = set(t.lower() for t in tags)
    for dp in user_dietary_prefs:
        if dp.lower() in tag_set:
            pref_score += 0.1
        else:
            pref_score -= 0.15
    pref_score = max(0, min(1, pref_score))
    
    # 7. Popularity
    if rating_avg is not None and rating_count is not None:
        norm_rating = (rating_avg - 1) / 4
        count_factor = min(1, math.log(rating_count + 1) / math.log(100))
        pop_score = norm_rating * 0.7 + count_factor * 0.3
    else:
        pop_score = 0.5
    
    # 8. Diversity penalty
    div_penalty = 0
    if recipe_id in recent_recipe_ids:
        div_penalty += 0.6
    if cuisine and recent_cuisines:
        cuisine_count = sum(1 for c in recent_cuisines if c.lower() == cuisine.lower())
        div_penalty += min(0.3, cuisine_count * 0.1)
    div_penalty = max(0, min(1, div_penalty))
    
    # Weighted sum
    total_score = (
        w['ingredientMatch'] * match_ratio +
        w['quantitySufficiency'] * qty_score +
        w['expiryUrgency'] * expiry_score +
        w['pantryStaple'] * staple_adj +
        w['substitution'] * sub_credit +
        w['preference'] * pref_score +
        w['popularity'] * pop_score +
        w['diversity'] * (1 - div_penalty)
    ) * 100
    total_score = max(0, min(100, total_score))
    
    return {
        'recipe_id': recipe_id,
        'title': recipe_title,
        'total_score': round(total_score, 2),
        'match_ratio': round(match_ratio, 3),
        'matched': len(matched),
        'missing': len(missing),
        'expiry_boost': round(expiry_score, 3),
        'pref_score': round(pref_score, 3),
        'pop_score': round(pop_score, 3),
        'div_penalty': round(div_penalty, 3),
    }

print('score_recipe function defined')

In [ ]:
# Load data and score all recipes for the test user

# Load pantry
pantry_data = sb.table('pantry_items').select('name, standardized_ingredient_id, expiry_date').eq('user_id', TEST_USER_ID).execute()
pantry_names = [p['name'] for p in pantry_data.data if p.get('name')]

now = datetime.utcnow()
seven_days = now + timedelta(days=7)
pantry_expiring = [
    p['name'] for p in pantry_data.data
    if p.get('expiry_date') and now <= datetime.fromisoformat(p['expiry_date'].replace('Z', '+00:00')).replace(tzinfo=None) <= seven_days
]

# Load recipes with ingredients
recipes_data = sb.table('recipes').select('id, title, cuisine_name, rating_avg, rating_count, tags').is_('deleted_at', 'null').execute()

ri_data = sb.table('recipe_ingredients').select(
    'recipe_id, display_name, standardized_ingredients(canonical_name)'
).is_('deleted_at', 'null').execute()

# Group ingredients by recipe
ingredients_by_recipe = defaultdict(list)
for row in ri_data.data:
    name = row.get('display_name', '')
    si = row.get('standardized_ingredients')
    if si and isinstance(si, dict):
        name = si.get('canonical_name', name)
    ingredients_by_recipe[row['recipe_id']].append(name)

# Score all recipes
results = []
for r in recipes_data.data:
    ing_names = ingredients_by_recipe.get(r['id'], [])
    if not ing_names:
        continue
    result = score_recipe(
        recipe_id=r['id'],
        recipe_title=r['title'],
        ingredient_names=ing_names,
        pantry_names=pantry_names,
        pantry_expiring=pantry_expiring,
        cuisine=r.get('cuisine_name'),
        rating_avg=r.get('rating_avg'),
        rating_count=r.get('rating_count'),
        tags=r.get('tags', []),
    )
    results.append(result)

df = pd.DataFrame(results).sort_values('total_score', ascending=False)
print(f'Scored {len(df)} recipes')
display(df.head(20))

## 4.4 Weight Sensitivity Analysis

In [ ]:
# Sweep each weight parameter and observe how top-10 changes

weight_keys = list(DEFAULT_WEIGHTS.keys())
sweep_values = np.linspace(0.1, 0.8, 8)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, key in enumerate(weight_keys):
    top10_scores = []
    for val in sweep_values:
        # Adjust the swept weight, redistribute remaining evenly
        w = dict(DEFAULT_WEIGHTS)
        remaining = 1.0 - val
        other_keys = [k for k in weight_keys if k != key]
        for ok in other_keys:
            w[ok] = remaining / len(other_keys)
        w[key] = val
        
        scores = []
        for r in recipes_data.data:
            ing_names = ingredients_by_recipe.get(r['id'], [])
            if not ing_names:
                continue
            result = score_recipe(
                recipe_id=r['id'],
                recipe_title=r['title'],
                ingredient_names=ing_names,
                pantry_names=pantry_names,
                pantry_expiring=pantry_expiring,
                cuisine=r.get('cuisine_name'),
                rating_avg=r.get('rating_avg'),
                rating_count=r.get('rating_count'),
                weights=w,
            )
            scores.append(result['total_score'])
        
        scores.sort(reverse=True)
        top10_scores.append(np.mean(scores[:10]) if len(scores) >= 10 else np.mean(scores))
    
    axes[idx].plot(sweep_values, top10_scores, 'o-')
    axes[idx].set_title(f'{key}')
    axes[idx].set_xlabel('Weight value')
    axes[idx].set_ylabel('Avg top-10 score')

plt.suptitle('Weight Sensitivity: Average Top-10 Score vs Weight Value', fontsize=14)
plt.tight_layout()
plt.show()

## 4.5 Score Distribution Visualization

In [ ]:
# Histogram of total scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Total score histogram
axes[0].hist(df['total_score'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Total Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Recipe Scores')

# 2. Scatter: match ratio vs total score
axes[1].scatter(df['match_ratio'], df['total_score'], alpha=0.5, s=20)
axes[1].set_xlabel('Ingredient Match Ratio')
axes[1].set_ylabel('Total Score')
axes[1].set_title('Match Ratio vs Total Score')

# 3. Add cuisine data for box plot
recipe_map = {r['id']: r for r in recipes_data.data}
df['cuisine'] = df['recipe_id'].map(lambda rid: recipe_map.get(rid, {}).get('cuisine_name', 'unknown'))
top_cuisines = df['cuisine'].value_counts().head(8).index
df_top = df[df['cuisine'].isin(top_cuisines)]
if not df_top.empty:
    df_top.boxplot(column='total_score', by='cuisine', ax=axes[2], rot=45)
    axes[2].set_title('Scores by Cuisine')
    axes[2].set_xlabel('Cuisine')
    axes[2].set_ylabel('Total Score')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4.6 Edge Case Testing

In [ ]:
# Test 1: Empty pantry
print('=== Edge Case: Empty Pantry ===')
empty_results = []
for r in recipes_data.data[:50]:
    ing_names = ingredients_by_recipe.get(r['id'], [])
    if not ing_names:
        continue
    result = score_recipe(
        recipe_id=r['id'], recipe_title=r['title'],
        ingredient_names=ing_names,
        pantry_names=[], pantry_expiring=[],
    )
    empty_results.append(result)

empty_df = pd.DataFrame(empty_results).sort_values('total_score', ascending=False)
print(f'Max score with empty pantry: {empty_df["total_score"].max():.1f}')
print(f'Mean score with empty pantry: {empty_df["total_score"].mean():.1f}')
print(f'All match_ratios are 0: {(empty_df["match_ratio"] == 0).all()}')
print()

# Test 2: Fully stocked pantry (every ingredient available)
print('=== Edge Case: Fully Stocked Pantry ===')
all_ingredient_names = set()
for names in ingredients_by_recipe.values():
    all_ingredient_names.update(names)

full_results = []
for r in recipes_data.data[:50]:
    ing_names = ingredients_by_recipe.get(r['id'], [])
    if not ing_names:
        continue
    result = score_recipe(
        recipe_id=r['id'], recipe_title=r['title'],
        ingredient_names=ing_names,
        pantry_names=list(all_ingredient_names), pantry_expiring=[],
    )
    full_results.append(result)

full_df = pd.DataFrame(full_results).sort_values('total_score', ascending=False)
print(f'Max score with full pantry: {full_df["total_score"].max():.1f}')
print(f'Mean score with full pantry: {full_df["total_score"].mean():.1f}')
print(f'All match_ratios are 1: {(full_df["match_ratio"] == 1.0).all()}')
print()

# Test 3: Many expiring items
print('=== Edge Case: Many Expiring Items ===')
expiry_results = []
for r in recipes_data.data[:50]:
    ing_names = ingredients_by_recipe.get(r['id'], [])
    if not ing_names:
        continue
    result = score_recipe(
        recipe_id=r['id'], recipe_title=r['title'],
        ingredient_names=ing_names,
        pantry_names=list(all_ingredient_names),
        pantry_expiring=list(all_ingredient_names),  # everything is expiring!
    )
    expiry_results.append(result)

expiry_df = pd.DataFrame(expiry_results).sort_values('total_score', ascending=False)
print(f'Max score with all expiring: {expiry_df["total_score"].max():.1f}')
print(f'Mean expiry_boost: {expiry_df["expiry_boost"].mean():.3f}')
print(f'Expiry boost is higher than full pantry (no expiry):')
print(f'  Mean total_score (expiring): {expiry_df["total_score"].mean():.1f}')
print(f'  Mean total_score (not expiring): {full_df["total_score"].mean():.1f}')
print()

# Test 4: Substitution with known pair
print('=== Edge Case: Substitution (lemon -> lime) ===')
sub_result = score_recipe(
    recipe_id='test', recipe_title='Lemon Chicken',
    ingredient_names=['chicken breast', 'lemon', 'garlic'],
    pantry_names=['chicken breast', 'lime', 'garlic'],
    pantry_expiring=[],
)
print(f'Score with lime substituting lemon: {sub_result["total_score"]:.1f}')
print(f'Match ratio (lemon not directly matched): {sub_result["match_ratio"]:.3f}')

## 4.7 A/B Weight Comparison

In [ ]:
# Compare two weight configurations side by side

config_a = dict(DEFAULT_WEIGHTS)  # baseline
config_b = dict(DEFAULT_WEIGHTS)
config_b['ingredientMatch'] = 0.25
config_b['expiryUrgency'] = 0.25  # prioritise using up expiring items

def score_all(weights):
    scored = []
    for r in recipes_data.data:
        ing_names = ingredients_by_recipe.get(r['id'], [])
        if not ing_names:
            continue
        result = score_recipe(
            recipe_id=r['id'], recipe_title=r['title'],
            ingredient_names=ing_names,
            pantry_names=pantry_names,
            pantry_expiring=pantry_expiring,
            cuisine=r.get('cuisine_name'),
            rating_avg=r.get('rating_avg'),
            rating_count=r.get('rating_count'),
            weights=weights,
        )
        scored.append(result)
    return pd.DataFrame(scored).sort_values('total_score', ascending=False)

df_a = score_all(config_a)
df_b = score_all(config_b)

# Compare top 10
print('=== Config A (Default Weights) — Top 10 ===')
display(df_a[['title', 'total_score', 'match_ratio']].head(10))

print()
print('=== Config B (Higher Expiry Weight) — Top 10 ===')
display(df_b[['title', 'total_score', 'match_ratio']].head(10))

# Show which recipes moved up/down
merged = df_a[['recipe_id', 'total_score']].merge(
    df_b[['recipe_id', 'total_score']],
    on='recipe_id', suffixes=('_a', '_b')
)
merged['delta'] = merged['total_score_b'] - merged['total_score_a']
merged = merged.sort_values('delta', ascending=False)

print()
print('=== Biggest movers (B vs A) ===')
print('Top 5 gainers:')
display(merged.head(5))
print('Top 5 losers:')
display(merged.tail(5))